# ⚡ GridGuard — PySpark Electricity Consumption Anomaly Detection

**GridGuard: A PySpark-Based Electricity Consumption Anomaly Detection and Visualization System**

This notebook runs the complete pipeline **top to bottom in Google Colab**:

```
Raw CSV ─► [HDFS raw storage] ─► PySpark cleaning ─► hourly/daily aggregation
        ─► rolling z-score anomaly detection ─► peak analysis
        ─► [HDFS] processed outputs ─► export ─► Streamlit dashboard (app.py)
```

**Dataset:** UCI *Individual Household Electric Power Consumption* (real data, 1-minute readings,
one household, Dec 2006 – Nov 2010). Primary metric: `Global_active_power` (kW).
No household IDs or anomaly labels exist in this dataset and none are invented here.

### Two explicit storage modes
| Mode | `USE_HDFS` | What happens |
|---|---|---|
| **A – HDFS enabled** | `True` | Connects to a *reachable* Hadoop NameNode; uploads the raw file to HDFS, reads it back, writes results to HDFS. |
| **B – Colab standalone** | `False` (default) | Spark runs locally in Colab; results are saved on the Colab disk. **Never labelled as HDFS.** |

> Colab does **not** include a running HDFS cluster. If you choose Mode A but the NameNode cannot be
> reached, the notebook says so and falls back to Mode B. The banner printed in section 3 always shows the mode that is *actually* active.

**Runtime:** Runtime ▸ Run all. The full dataset (≈127 MB, ≈2 million rows) takes a few minutes.

## 1. Configuration

In [ ]:
import os, sys, time, socket, shutil, zipfile, urllib.request
from datetime import datetime, timedelta, timezone
from urllib.parse import urlparse

# ------------------------------- storage mode ---------------------------------------
USE_HDFS          = False                       # True = Mode A (HDFS), False = Mode B (standalone)
HDFS_NAMENODE_URI = "hdfs://<namenode-host>:8020"   # e.g. hdfs://my-vm.example.com:8020 (must be reachable FROM COLAB)
HDFS_BASE_DIR     = "/gridguard"
HDFS_USER         = "hadoop"

# ------------------------------- anomaly detection -----------------------------------
Z_THRESHOLD       = 3.0     # flag when |z| >= threshold
BASELINE_DAYS     = 14      # compare with the same hour on the previous N days
MIN_HISTORY       = 5       # minimum prior observations needed to score (>= 2)
MIN_VALID_MINUTES = 30      # an hour needs at least this many valid minute readings
TOP_N_PEAKS       = 50

# ------------------------------- paths ------------------------------------------------
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
DATA_DIR   = f"{BASE}/gridguard_data"
OUTPUT_DIR = f"{BASE}/gridguard_outputs"      # local outputs (also the dashboard export)
DATA_FILE  = "household_power_consumption.txt"
RAW_PATH   = f"{DATA_DIR}/{DATA_FILE}"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True)

assert MIN_HISTORY >= 2 and MIN_HISTORY <= BASELINE_DAYS, "need 2 <= MIN_HISTORY <= BASELINE_DAYS"
print("Configuration OK")

## 2. Dependency installation

In [ ]:
import subprocess
def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

# PySpark needs Java (Colab usually has it; install if missing)
if shutil.which("java") is None:
    print("Installing Java ...")
    run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless")
_v = run("java -version"); print((_v.stderr or _v.stdout or "Java NOT found - install it before continuing").splitlines()[0])

# Python packages (pandas is only used to plot small, already-aggregated results)
r = run(f'{sys.executable} -m pip -q install "pyspark>=3.5.1,<4.0" plotly')
print("pip finished with code", r.returncode)
if r.returncode != 0: print(r.stderr[-800:])
import pyspark; print("PySpark", pyspark.__version__)

## 3. Dataset download and extraction
The file is downloaded from the official UCI repository. If the download fails (network, UCI downtime), upload
`household_power_consumption.txt` manually via the Colab **Files** panel into the folder shown below, then re-run this cell.
The original ZIP/TXT is never modified.

In [ ]:
DATASET_URLS = [
    "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip",
]
EXPECTED_HEADER = "Date;Time;Global_active_power;Global_reactive_power;Voltage;Global_intensity;Sub_metering_1;Sub_metering_2;Sub_metering_3"

def header_ok(path):
    if not os.path.isfile(path): return False
    with open(path, encoding="utf-8", errors="replace") as fh:
        return fh.readline().strip().lstrip("\ufeff") == EXPECTED_HEADER

if header_ok(RAW_PATH):
    print("Dataset already present:", RAW_PATH)
else:
    zip_path = f"{DATA_DIR}/dataset.zip"
    for url in DATASET_URLS:
        try:
            print("Downloading", url)
            urllib.request.urlretrieve(url, zip_path)
            with zipfile.ZipFile(zip_path) as zf:
                member = [m for m in zf.namelist() if m.endswith(DATA_FILE)][0]
                with zf.open(member) as src, open(RAW_PATH, "wb") as out:
                    shutil.copyfileobj(src, out)
            break
        except Exception as exc:
            print("  failed:", exc)
    if not header_ok(RAW_PATH):
        raise SystemExit(f"""
Could not obtain the dataset automatically.
Manual option: download it from
https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption ,
unzip it, upload '{DATA_FILE}' to {DATA_DIR} (Colab Files panel), and re-run this cell.""")
print(f"Ready: {RAW_PATH}  ({os.path.getsize(RAW_PATH)/1e6:.1f} MB)")

## 4. PySpark session initialization and storage mode

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

use_hdfs_now = False
hdfs_reason = "HDFS not enabled (USE_HDFS=False) - Mode B: Colab standalone"
if USE_HDFS:
    u = urlparse(HDFS_NAMENODE_URI)
    try:
        socket.create_connection((u.hostname, u.port or 8020), timeout=4).close()
        use_hdfs_now, hdfs_reason = True, f"NameNode {u.hostname}:{u.port or 8020} reachable - Mode A: HDFS"
    except Exception as exc:
        hdfs_reason = f"HDFS requested but NOT reachable ({exc}) - falling back to Mode B (local disk)"

if use_hdfs_now:
    os.environ["HADOOP_USER_NAME"] = HDFS_USER

builder = (SparkSession.builder.appName("GridGuard").master("local[*]")
           .config("spark.driver.memory", "3g")
           .config("spark.sql.shuffle.partitions", "8")
           .config("spark.sql.session.timeZone", "UTC")          # no DST shifts of meter wall-clock time
           .config("spark.sql.ansi.enabled", "false")            # bad casts -> NULL
           .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
           .config("spark.ui.showConsoleProgress", "false"))
if use_hdfs_now:
    builder = (builder.config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
               .config("spark.hadoop.ipc.client.connect.max.retries", "2")
               .config("spark.hadoop.ipc.client.connect.timeout", "5000"))
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

if use_hdfs_now:
    STORAGE_MODE, STORAGE_ROOT = "hdfs", f"{HDFS_NAMENODE_URI.rstrip('/')}{HDFS_BASE_DIR}"
else:
    STORAGE_MODE, STORAGE_ROOT = "local", OUTPUT_DIR

from IPython.display import HTML, display
color = "#1a7f37" if STORAGE_MODE == "hdfs" else "#b35900"
label = "HDFS" if STORAGE_MODE == "hdfs" else "LOCAL FILESYSTEM (not HDFS)"
display(HTML(f'<div style="padding:10px;border-left:6px solid {color};background:#f6f8fa">'
             f'<b>Active storage mode: {label}</b><br>root: <code>{STORAGE_ROOT}</code><br>{hdfs_reason}</div>'))
print("Spark", spark.version)

def path_of(rel, root=None):
    return f"{(root or STORAGE_ROOT).rstrip('/')}/{rel.strip('/')}"

## 5. HDFS integration (Mode A) / local storage (Mode B)
Directory layout (identical under `/gridguard` in HDFS and under the local output folder):
`raw/`, `processed/hourly_consumption/`, `results/detected_anomalies/`, `results/peak_analysis/…`
(plus `processed/daily_consumption/` and `processed/pipeline_summary/`, two small additions the dashboard needs).

**Connecting to a real HDFS cluster from Colab:** set `USE_HDFS=True` and `HDFS_NAMENODE_URI` to a NameNode that is
reachable *from Google's cloud* (public IP/hostname with ports 8020 for the NameNode and 9866 for DataNodes open, DataNodes advertising
a resolvable hostname). `localhost` in Colab means the Colab VM, so a Docker cluster on your laptop is **not** reachable from Colab.
For a laptop cluster, run `run_pipeline.py --hdfs` locally instead (see `hadoop/README.md`). Only expose a Hadoop cluster to the internet if you understand the security implications - this dev setup has no authentication.

In [ ]:
HDFS_DIRS = ["raw", "processed/hourly_consumption", "processed/daily_consumption", "processed/pipeline_summary",
             "results/detected_anomalies", "results/peak_analysis/top_periods", "results/peak_analysis/peak_hour_frequency"]

def _hadoop_fs():
    jvm = spark._jvm
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI.create(HDFS_NAMENODE_URI), spark._jsc.hadoopConfiguration())
    return fs, jvm.org.apache.hadoop.fs.Path

def ensure_directories():
    if STORAGE_MODE == "hdfs":
        fs, P = _hadoop_fs()
        for rel in HDFS_DIRS: fs.mkdirs(P(path_of(rel)))
    else:
        for rel in HDFS_DIRS[1:]: os.makedirs(path_of(rel), exist_ok=True)
    print(f"Directory structure ready in {STORAGE_MODE.upper()} storage under {STORAGE_ROOT}")

def upload_raw(local_file):
    """Mode A: copy the raw file into HDFS /raw and return the HDFS path. Mode B: read the file where it is."""
    if STORAGE_MODE != "hdfs": return local_file
    fs, P = _hadoop_fs()
    dest = path_of(f"raw/{os.path.basename(local_file)}")
    fs.copyFromLocalFile(False, True, P("file://" + local_file), P(dest))
    assert fs.exists(P(dest)), "upload failed"
    print("Uploaded raw dataset to", dest)
    return dest

def verify_table(rel, root=None, mode=None):
    mode = mode or STORAGE_MODE
    p = path_of(rel, root); files = []
    if mode == "hdfs":
        fs, P = _hadoop_fs()
        if fs.exists(P(p)): files = [(s.getPath().getName(), int(s.getLen())) for s in fs.listStatus(P(p))]
    elif os.path.isdir(p):
        files = [(f, os.path.getsize(os.path.join(p, f))) for f in os.listdir(p) if os.path.isfile(os.path.join(p, f))]
    ok = any(n == "_SUCCESS" for n, _ in files) and any(n.startswith("part-") and s > 0 for n, s in files)
    return {"path": p, "storage": mode, "ok": ok, "bytes": sum(s for n, s in files if n.startswith("part-"))}

def write_table(df, rel, root=None):
    p = path_of(rel, root)
    (df.coalesce(1).write.mode("overwrite").option("header", "true")
       .option("timestampFormat", "yyyy-MM-dd HH:mm:ss").csv(p))
    return p

ensure_directories()
READ_PATH = upload_raw(RAW_PATH)       # in Mode A this is an hdfs:// path: Spark now reads FROM HDFS
print("Spark will read the raw data from:", READ_PATH)

## 6. Dataset loading and schema inspection

In [ ]:
RAW_COLUMNS = ["Date", "Time", "Global_active_power", "Global_reactive_power", "Voltage",
               "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
RAW_SCHEMA = StructType([StructField(c, StringType(), True) for c in RAW_COLUMNS])

raw_df = spark.read.csv(READ_PATH, schema=RAW_SCHEMA, sep=";", header=True, mode="PERMISSIVE")
print("Raw schema (all strings on purpose - no silent type guessing):")
raw_df.printSchema()
raw_df.show(5, truncate=False)
print("Partitions:", raw_df.rdd.getNumPartitions())

## 7. Data cleaning and missing-value treatment
* `?`, empty strings, text, `NaN`, negative or implausible values → `NULL` (never silently replaced by 0).
* Missing power readings are **kept as NULL** (not imputed) so averages ignore them and we can count valid minutes per hour.
* Rows with an unparseable timestamp are dropped; duplicate timestamps are de-duplicated.

In [ ]:
NUMERIC = {"Global_active_power": "global_active_power", "Global_reactive_power": "global_reactive_power",
           "Voltage": "voltage", "Global_intensity": "global_intensity",
           "Sub_metering_1": "sub_metering_1", "Sub_metering_2": "sub_metering_2", "Sub_metering_3": "sub_metering_3"}
MAX_PLAUSIBLE = 1_000_000.0

def sanitise(col):
    parsed = F.trim(col).cast("double")                   # '?' and text -> NULL
    return F.when(parsed.between(0.0, MAX_PLAUSIBLE), parsed)   # negatives, NaN, Infinity -> NULL

# ---- timestamp creation (day-first format d/M/yyyy) ----
parsed = raw_df.withColumn("timestamp", F.to_timestamp(
    F.concat_ws(" ", F.trim("Date"), F.trim("Time")), "d/M/yyyy HH:mm:ss"))
parsed = parsed.withColumn("_gap_parsed", F.trim("Global_active_power").cast("double"))
for src, dst in NUMERIC.items():
    parsed = parsed.withColumn(dst, sanitise(F.col(src)))
parsed = parsed.select("timestamp", "_gap_parsed", *NUMERIC.values()).persist()

quality = parsed.agg(
    F.count(F.lit(1)).alias("total_records"),
    F.sum(F.when(F.col("timestamp").isNull(), 1).otherwise(0)).alias("invalid_timestamp_rows"),
    F.sum(F.when(F.col("_gap_parsed").isNull(), 1).otherwise(0)).alias("missing_gap_rows"),
    F.sum(F.when(F.col("_gap_parsed").isNotNull() & F.col("global_active_power").isNull(), 1).otherwise(0)).alias("invalid_gap_rows"),
    F.sum(F.when(F.col("global_active_power").isNotNull(), 1).otherwise(0)).alias("valid_gap_rows"),
).collect()[0].asDict()
quality = {k: int(v or 0) for k, v in quality.items()}
print("DATA QUALITY (computed from the data just read):")
for k, v in quality.items(): print(f"  {k:24s} {v:>12,}")
if quality["total_records"]:
    print(f"  missing share            {quality['missing_gap_rows']/quality['total_records']*100:>11.2f} %")

clean_df = (parsed.drop("_gap_parsed").filter(F.col("timestamp").isNotNull())
            .dropDuplicates(["timestamp"]))
clean_df.printSchema()

## 8. Timestamp features and hourly aggregation (PySpark)
`observed_kw` = mean of the valid minute readings in the hour (kW). Hours with fewer than `MIN_VALID_MINUTES` valid minutes
get `NULL` so a mostly-missing hour is never treated as a real measurement.

In [ ]:
hourly_df = (clean_df
    .withColumn("hour_start", F.date_trunc("hour", "timestamp"))
    .groupBy("hour_start").agg(
        F.count(F.lit(1)).alias("total_readings"),
        F.count("global_active_power").alias("valid_readings"),
        F.avg("global_active_power").alias("_avg_kw"),
        F.max("global_active_power").alias("max_minute_kw"),
        F.min("global_active_power").alias("min_minute_kw"),
        F.avg("voltage").alias("avg_voltage"))
    .withColumnRenamed("hour_start", "timestamp")
    .withColumn("observed_kw", F.when(F.col("valid_readings") >= MIN_VALID_MINUTES, F.round("_avg_kw", 4)))
    .drop("_avg_kw")
    .withColumn("date", F.to_date("timestamp"))
    .withColumn("hour_of_day", F.hour("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))          # 1 = Sunday
    .withColumn("day_name", F.date_format("timestamp", "E"))
    .withColumn("is_weekend", F.dayofweek("timestamp").isin(1, 7))
    .withColumn("max_minute_kw", F.round("max_minute_kw", 4))
    .withColumn("min_minute_kw", F.round("min_minute_kw", 4))
    .withColumn("avg_voltage", F.round("avg_voltage", 3))
    .select("timestamp", "date", "hour_of_day", "day_of_week", "day_name", "is_weekend", "observed_kw",
            "max_minute_kw", "min_minute_kw", "valid_readings", "total_readings", "avg_voltage")
    .orderBy("timestamp")).persist()
print("Hourly rows:", hourly_df.count())
hourly_df.show(5, truncate=False)

# ---- daily aggregation (Spark SQL) ----
hourly_df.createOrReplaceTempView("hourly")
daily_preview = spark.sql("""
    SELECT date, ROUND(AVG(observed_kw), 4) AS avg_kw, ROUND(MAX(observed_kw), 4) AS peak_hourly_kw,
           COUNT(observed_kw) AS hours_with_data
    FROM hourly GROUP BY date ORDER BY date""")
daily_preview.show(5)

## 9. Historical baselines — Spark SQL + window functions
For every hourly row the baseline uses **only the same hour-of-day on the previous `BASELINE_DAYS` calendar days**:

`RANGE BETWEEN BASELINE_DAYS PRECEDING AND 1 PRECEDING` over the day number. The current day is never inside its own
window, so there is **no look-ahead leakage**, and gaps in the data do not stretch the window.

In [ ]:
hourly_df.withColumn("date_num", F.datediff(F.col("date"), F.to_date(F.lit("1970-01-01")))) \
         .createOrReplaceTempView("hourly_features")

baseline_df = spark.sql(f"""
    SELECT *,
           COUNT(observed_kw)        OVER w AS history_count,
           AVG(observed_kw)          OVER w AS baseline_mean_kw,
           STDDEV_SAMP(observed_kw)  OVER w AS baseline_std_kw
    FROM hourly_features
    WINDOW w AS (PARTITION BY hour_of_day ORDER BY date_num
                 RANGE BETWEEN {BASELINE_DAYS} PRECEDING AND 1 PRECEDING)
""").drop("date_num")
baseline_df.select("timestamp", "observed_kw", "history_count", "baseline_mean_kw", "baseline_std_kw") \
           .where("history_count >= 5").show(5)

## 10. Statistical anomaly detection and explanations
**z = (observed − baseline_mean) / baseline_std** (sample standard deviation). Flag when `|z| ≥ Z_THRESHOLD`.

Safety rules: missing/negative observation → *missing_observation*; fewer than `MIN_HISTORY` prior points → *insufficient_history*;
zero or undefined std (constant history) → *zero_variance*. In all three cases no z-score is computed and nothing is flagged.
Severity bands are relative to the threshold: Moderate (threshold…+1), Strong (+1…+2), Extreme (≥ +2).

> A flag is a **statistical deviation only**. It does not prove electricity theft, equipment failure, or any fault.

In [ ]:
ZERO_STD_EPS = 1e-9
DISCLAIMER = ("This is a statistical deviation only and does not by itself indicate theft, "
              "equipment failure, or a fault.")

obs, mean, std, n = F.col("observed_kw"), F.col("baseline_mean_kw"), F.col("baseline_std_kw"), F.col("history_count")
status = (F.when(obs.isNull(), "missing_observation")
           .when(n < MIN_HISTORY, "insufficient_history")
           .when(std.isNull() | (std <= ZERO_STD_EPS), "zero_variance")
           .otherwise("ok"))
scored = baseline_df.withColumn("baseline_status", status)
ok = F.col("baseline_status") == "ok"
scored = (scored
    .withColumn("z_score", F.when(ok, F.round((obs - mean) / std, 4)))
    .withColumn("abs_z_score", F.abs("z_score"))
    .withColumn("is_anomaly", F.coalesce(F.col("abs_z_score") >= Z_THRESHOLD, F.lit(False)))
    .withColumn("anomaly_direction",
                F.when(~ok, "Not evaluated")
                 .when(F.col("is_anomaly") & (F.col("z_score") > 0), "High")
                 .when(F.col("is_anomaly") & (F.col("z_score") < 0), "Low")
                 .otherwise("Normal"))
    .withColumn("severity",
                F.when(~F.col("is_anomaly"), "None")
                 .when(F.col("abs_z_score") >= Z_THRESHOLD + 2, "Extreme")
                 .when(F.col("abs_z_score") >= Z_THRESHOLD + 1, "Strong")
                 .otherwise("Moderate")))

base_text = ("compared with the historical baseline for this time of day "
             "(baseline mean %.2f kW, std %.2f kW over %d prior observations; z-score %.2f). ")
scored = scored.withColumn("explanation",
    F.when(F.col("anomaly_direction") == "High",
           F.format_string("Consumption of %.2f kW is unusually high " + base_text + DISCLAIMER, obs, mean, std, n, F.col("z_score")))
     .when(F.col("anomaly_direction") == "Low",
           F.format_string("Consumption of %.2f kW is unusually low " + base_text + DISCLAIMER, obs, mean, std, n, F.col("z_score")))
     .when(F.col("anomaly_direction") == "Normal",
           F.format_string("Within the expected range for this time of day (z-score %.2f).", F.col("z_score")))
     .when(F.col("baseline_status") == "insufficient_history",
           F.format_string("Not evaluated: only %d prior observation(s) available; at least " + str(MIN_HISTORY) + " are required.", n))
     .when(F.col("baseline_status") == "zero_variance",
           F.lit("Not evaluated: the historical baseline has zero variance, so a z-score is undefined."))
     .otherwise(F.lit("Not evaluated: no valid observation for this hour (too few valid minute readings).")))
scored = (scored.withColumn("baseline_mean_kw", F.round("baseline_mean_kw", 4))
                .withColumn("baseline_std_kw", F.round("baseline_std_kw", 4))).persist()

stats = scored.agg(F.count(F.lit(1)).alias("hourly_rows"),
                   F.sum(F.when(ok, 1).otherwise(0)).alias("evaluated_hours"),
                   F.sum(F.col("is_anomaly").cast("int")).alias("anomaly_count"),
                   F.min("timestamp").alias("data_start"), F.max("timestamp").alias("data_end")).collect()[0]
print(f"Hourly rows: {stats['hourly_rows']:,} | scored: {stats['evaluated_hours'] or 0:,} | anomalies: {stats['anomaly_count'] or 0:,}")
print("Status breakdown:"); scored.groupBy("baseline_status").count().orderBy(F.desc("count")).show()

ANOMALY_COLUMNS = ["timestamp", "date", "hour_of_day", "day_name", "observed_kw", "baseline_mean_kw", "baseline_std_kw",
                   "history_count", "z_score", "abs_z_score", "anomaly_direction", "severity", "baseline_status", "explanation"]
anomalies_df = scored.filter("is_anomaly").select(*ANOMALY_COLUMNS).orderBy(F.desc("abs_z_score"))
print("Strongest anomalies (explanations shown in full):")
anomalies_df.select("timestamp", "observed_kw", "baseline_mean_kw", "z_score", "anomaly_direction", "severity").show(10, truncate=False)
for r in anomalies_df.select("explanation").limit(3).collect(): print("•", r["explanation"])

## 11. Daily aggregation with anomaly counts and peak-demand analysis
* `top_periods` – the highest hourly demand periods with their historical baseline.
* `peak_hour_frequency` – for each hour of the day, on how many days it was the day's highest-demand hour (**recurring peaks**).

In [ ]:
daily_df = (scored.groupBy("date").agg(
        F.round(F.avg("observed_kw"), 4).alias("avg_kw"),
        F.round(F.max("observed_kw"), 4).alias("peak_hourly_kw"),
        F.round(F.min("observed_kw"), 4).alias("min_hourly_kw"),
        F.round(F.sum("observed_kw"), 3).alias("energy_kwh"),          # sum of hourly mean kW x 1 h (valid hours only)
        F.count("observed_kw").alias("hours_with_data"),
        F.sum(F.col("is_anomaly").cast("int")).alias("anomaly_count"))
    .withColumn("day_name", F.date_format("date", "E")).orderBy("date"))

peak_top_df = (scored.filter(obs.isNotNull()).orderBy(F.desc("observed_kw")).limit(TOP_N_PEAKS)
    .select("timestamp", "date", "hour_of_day", "day_name", "observed_kw", "baseline_mean_kw", "baseline_std_kw",
            "history_count", "z_score",
            F.round(obs - mean, 4).alias("diff_from_baseline_kw"),
            F.round(F.when(mean > 0, (obs - mean) / mean * 100), 2).alias("diff_from_baseline_pct"))
    .orderBy(F.desc("observed_kw")))

w_day = Window.partitionBy("date").orderBy(F.desc("observed_kw"), F.asc("hour_of_day"))
daily_peaks = scored.filter(obs.isNotNull()).withColumn("_rank", F.row_number().over(w_day)).filter("_rank = 1")
total_days = daily_peaks.count()
peak_freq_df = (daily_peaks.groupBy("hour_of_day").agg(F.count(F.lit(1)).alias("days_as_daily_peak"),
                                                       F.round(F.avg("observed_kw"), 4).alias("avg_peak_kw"))
    .withColumn("share_of_days_pct", F.round(F.col("days_as_daily_peak") * 100.0 / F.lit(max(total_days, 1)), 2))
    .orderBy(F.desc("days_as_daily_peak")))

print("Top peak periods:"); peak_top_df.show(5, truncate=False)
print("Recurring daily-peak hours:"); peak_freq_df.show(6)

## 12. Save processed datasets and anomaly reports
Results are written to the **active** storage (HDFS in Mode A, local disk in Mode B) and verified.
In Mode A a second copy is written to the local output folder so the Streamlit dashboard can read it without WebHDFS networking.
The `pipeline_summary` table has the same schema as `run_pipeline.py`, so `app.py` accepts it unchanged.

In [ ]:
SUMMARY_SCHEMA = ("total_records long, invalid_timestamp_rows long, missing_gap_rows long, invalid_gap_rows long, "
    "valid_gap_rows long, hourly_rows long, evaluated_hours long, anomaly_count long, anomaly_threshold double, "
    "baseline_days int, min_history int, min_valid_minutes int, baseline_by_day_type boolean, data_start string, "
    "data_end string, source_file string, storage_mode string, copy_location string, storage_root string, processed_at string")

info = {**quality,
        "hourly_rows": int(stats["hourly_rows"] or 0), "evaluated_hours": int(stats["evaluated_hours"] or 0),
        "anomaly_count": int(stats["anomaly_count"] or 0), "anomaly_threshold": float(Z_THRESHOLD),
        "baseline_days": int(BASELINE_DAYS), "min_history": int(MIN_HISTORY), "min_valid_minutes": int(MIN_VALID_MINUTES),
        "baseline_by_day_type": False, "data_start": str(stats["data_start"] or ""), "data_end": str(stats["data_end"] or ""),
        "source_file": str(READ_PATH), "storage_mode": STORAGE_MODE, "storage_root": STORAGE_ROOT,
        "processed_at": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")}

def summary_df(copy_location):
    row = dict(info, copy_location=copy_location)
    order = [c.split()[0] for c in SUMMARY_SCHEMA.split(", ")]
    return spark.createDataFrame([tuple(row[c] for c in order)], schema=SUMMARY_SCHEMA)

TABLES = [("processed/hourly_consumption", scored), ("processed/daily_consumption", daily_df),
          ("results/detected_anomalies", anomalies_df), ("results/peak_analysis/top_periods", peak_top_df),
          ("results/peak_analysis/peak_hour_frequency", peak_freq_df)]

targets = [(STORAGE_ROOT, STORAGE_MODE, STORAGE_MODE)]
if STORAGE_MODE == "hdfs":
    targets.append((OUTPUT_DIR, "local", "local_export_of_hdfs"))
for root, mode, copy_location in targets:
    for rel in HDFS_DIRS[1:]:
        if mode == "local": os.makedirs(path_of(rel, root), exist_ok=True)
    for rel, df in TABLES: write_table(df, rel, root)
    write_table(summary_df(copy_location), "processed/pipeline_summary", root)

print("VERIFICATION (what was really written, and where):")
for root, mode, _ in targets:
    for rel in [t[0] for t in TABLES] + ["processed/pipeline_summary"]:
        v = verify_table(rel, root, mode)
        print(f"  {'OK  ' if v['ok'] else 'FAIL'} [{v['storage'].upper():5s}] {v['path']}  ({v['bytes']:,} bytes)")

## 13. Plot important results

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

daily_pd = daily_df.toPandas()
daily_pd["rolling_7d"] = daily_pd["avg_kw"].rolling(7, min_periods=1).mean()
fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_pd["date"], y=daily_pd["avg_kw"], name="Daily average", line=dict(width=1)))
fig.add_trace(go.Scatter(x=daily_pd["date"], y=daily_pd["rolling_7d"], name="7-day average", line=dict(width=3)))
fl = daily_pd[daily_pd["anomaly_count"] > 0]
fig.add_trace(go.Scatter(x=fl["date"], y=fl["avg_kw"], mode="markers", name="Days with anomalies", marker=dict(color="crimson", size=6)))
fig.update_layout(title="Daily consumption (kW)", template="plotly_white", height=420); fig.show()

# observed vs baseline around the strongest anomaly
top1 = anomalies_df.select("timestamp").limit(1).collect()
if top1:
    t0 = top1[0]["timestamp"]
    win = (scored.where(F.col("timestamp").between(F.lit(t0 - timedelta(hours=36)), F.lit(t0 + timedelta(hours=36))))
                 .orderBy("timestamp").toPandas())
    win["upper"] = win["baseline_mean_kw"] + Z_THRESHOLD * win["baseline_std_kw"]
    win["lower"] = win["baseline_mean_kw"] - Z_THRESHOLD * win["baseline_std_kw"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=win["timestamp"], y=win["upper"], line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=win["timestamp"], y=win["lower"], line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(45,123,255,0.18)", name=f"Baseline ± {Z_THRESHOLD:g}σ"))
    fig.add_trace(go.Scatter(x=win["timestamp"], y=win["baseline_mean_kw"], name="Historical mean", line=dict(dash="dash")))
    fig.add_trace(go.Scatter(x=win["timestamp"], y=win["observed_kw"], name="Observed", mode="lines+markers"))
    fig.update_layout(title="Strongest anomaly: observed vs. historical baseline (±36 h)", template="plotly_white", height=420); fig.show()
else:
    print("No anomalies met the rule with the current settings.")

prof = scored.groupBy("hour_of_day").agg(F.avg("observed_kw").alias("avg_kw")).orderBy("hour_of_day").toPandas()
px.line(prof, x="hour_of_day", y="avg_kw", markers=True, title="Average consumption by hour of day (kW)", template="plotly_white").show()
px.bar(peak_freq_df.orderBy("hour_of_day").toPandas(), x="hour_of_day", y="days_as_daily_peak",
       title="Recurring peak hours (days on which each hour was the daily peak)", template="plotly_white").show()

## 14. Export outputs for the Streamlit frontend
The dashboard (`app.py`) reads the folder layout written above. Download the archive and unzip it into the project's
`outputs/` folder (so that `outputs/processed/...` and `outputs/results/...` exist), then run `streamlit run app.py`.

In [ ]:
archive = shutil.make_archive(f"{BASE}/gridguard_outputs", "zip", OUTPUT_DIR)
print("Archive:", archive, f"({os.path.getsize(archive)/1e6:.2f} MB)")
print("Local output folder contents:")
for root_, dirs, files_ in os.walk(OUTPUT_DIR):
    for f in files_:
        if f.endswith(".csv"): print("  ", os.path.relpath(os.path.join(root_, f), OUTPUT_DIR))
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("(Not running in Colab - copy the archive manually.)")

## 15. Summary, limitations and conclusions

**What was done.** Raw minute-level readings were read with PySpark (from HDFS in Mode A, from the Colab disk in Mode B), cleaned
without imputation, aggregated to hourly/daily tables, compared with a leakage-free same-hour historical baseline using Spark SQL
window functions, scored with a rolling z-score, explained in plain language, analysed for peak demand, saved, verified and exported
for the dashboard. All numbers shown above were computed from the data in this run; nothing is hard-coded.

**Limitations**
* One household and one metric (`Global_active_power`); no ground-truth labels, so precision/recall **cannot** be measured.
* A z-score assumes the same-hour history is roughly stable and bell-shaped; consumption is skewed and seasonal (holidays, heating), so
  some flags are ordinary behaviour changes and some real irregularities may be missed.
* The baseline compares only with the previous `BASELINE_DAYS` days (no weekday/weekend split by default) and needs `MIN_HISTORY` prior points.
* Hours with too few valid minutes and constant baselines are *not evaluated*, not "normal".
* A flag is a statistical deviation, not evidence of theft or a fault.
* The dataset is small for a "big data" claim (~2 M rows); the pipeline is written with distributed constructs (DataFrames, window functions,
  HDFS) so the same code scales to larger multi-meter data, but this notebook runs Spark in local mode on one Colab VM.
* Mode A (real HDFS from Colab) requires a cluster reachable from Google's cloud; it was not exercised by the author of this notebook.

**Conclusion.** The notebook demonstrates the complete flow — storage, distributed processing, statistical detection, explanation and
export — and keeps storage claims honest through the explicit mode banner and file verification.